In [1]:
import numpy as np
import torch
from torch import nn
from torch.utils import data
import sys
sys.path.append('../')
from tools.SYNTHETIC import synthetic_data

true_w = torch.tensor([2, -3.4])
true_b = 4.2

features, labels = synthetic_data(true_w, true_b, 1000)

def load_array(data_arrays, batch_size, is_train=True):
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(2, 1)
        self.linear.weight.data.normal_(0, 0.01)
        self.linear.bias.data.fill_(0.0)

    def forward(self, x):
        return self.linear(x)

net = LinearRegression()

loss = nn.MSELoss()
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        y_hat = net(X)
        l = loss(y_hat, y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch+1}, loss {l:f}')

w = net.linear.weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net.linear.bias.data
print('b的估计误差：', true_b - b)

epoch 1, loss 0.000232
epoch 2, loss 0.000100
epoch 3, loss 0.000099
w的估计误差： tensor([-0.0007, -0.0004])
b的估计误差： tensor([-0.0004])
